## GPU ENGINE
[Rapids AI open beta Polars post](https://developer.nvidia.com/blog/polars-gpu-engine-powered-by-rapids-cudf-now-available-in-open-beta)

[Google Colab notebook demo from Rapids AI](https://colab.research.google.com/github/rapidsai-community/showcase/blob/main/accelerated_data_processing_examples/polars_gpu_engine_demo.ipynb?utm_source=nvidia+mktg&utm_medium=web&utm_campaign=polars+launch)


NOTE: Run this notebook manually in Colab or Kaggle; use the `Polars_GPU_XX.ipynb` notebooks locally. Large in-memory demos can exhaust GPU memory, so do not use Run All across these sections. After a manual kernel/runtime restart, rerun setup cell 3 and the chosen engine cell (8 or 14 for in-memory, 20 for streaming). Only the final multi-GPU section requires multiple GPUs.

In [ ]:
# Install into this notebook's kernel; keep installation errors visible.
%pip install "polars[gpu]==1.44.2" "cudf-polars-cu12>=26.10.2"
# Restart the runtime if requested, then rerun setup cell 3 and the chosen section's engine cell.

In [ ]:
# Only for Colab. In Kaggle just add dataset: https://www.kaggle.com/datasets/mihaiaftinescu/nyc-green-yellow
!mkdir data
!curl -L -o nyc-green-yellow.zip https://www.kaggle.com/api/v1/datasets/download/mihaiaftinescu/nyc-green-yellow
!unzip nyc-green-yellow.zip -d data/nyc-taxi

In [ ]:
import polars as pl
import os

zone_schema = {"LocationID": pl.Int32, "Borough": pl.String, "Zone": pl.String, "service_zone": pl.String}
# Modify path if using kaggle to input dataset
df_zones = pl.scan_csv("data/nyc-taxi/taxi_zone_lookup.csv", schema=zone_schema)

df = (
    pl.scan_parquet("data/nyc-taxi/yellow.parquet")
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)
df_green = (
    pl.scan_parquet("data/nyc-taxi/green.parquet")
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)

complex_keys = ["trip_distance", "DO_Zone", "DO_Borough", "PU_Borough"]


def taxi_summary(frame, keys):
    return frame.group_by(keys).agg(
        pl.len().alias("trip_count"),
        pl.sum("fare_amount").alias("total_revenue"),
        pl.mean("fare_amount").alias("avg_fare"),
        pl.mean("trip_distance").alias("avg_distance"),
        pl.mean("tip_amount").alias("avg_tip"),
    ).sort(keys).head()

In [ ]:
df_green.head().collect(engine="streaming")

## Streaming Engine

In [ ]:
%%time
taxi_summary(df_green, "trip_distance").collect(engine="streaming")

## GPU in-memory
The in-memory executor runs on a single GPU. This demo disables CUDA managed memory (UVM), so the query must fit in GPU memory. The streaming executor instead processes partitions and can spill buffers to host memory; that is separate from UVM.

In [ ]:
os.environ["POLARS_GPU_ENABLE_CUDA_MANAGED_MEMORY"] = "0"  # Forcing in-memory engine to not use UVM, just for the demo
gpu_in_mem_engine = pl.GPUEngine(executor="in-memory", raise_on_fail=True)

In [ ]:
%%time
# Note: This will fail if GPU memory is not large enough
taxi_summary(df_green, "trip_distance").collect(engine=gpu_in_mem_engine)

In [ ]:
# LazyFrame.profile is deprecated since polars 1.43 but remains available on this pinned 1.44.2 GPU stack.
# cudf-polars supports profiling only with the in-memory executor; CPU polars 2.x has no profile method.
result, prof = taxi_summary(df_green, "trip_distance").profile(engine=gpu_in_mem_engine)
prof.with_columns(((pl.col("end") - pl.col("start")) / pl.col("end").max() * 100).alias("pct_time_spent"))

## GPU in memory. More complex queries

In [ ]:
# If GPU memory remains allocated, restart the kernel/runtime manually, then rerun setup cell 3 and engine cell 14.

In [ ]:
%%time
taxi_summary(df_green, complex_keys).collect(engine="streaming")

In [ ]:
# Rerun this engine cell after a manual restart and setup cell 3.
os.environ["POLARS_GPU_ENABLE_CUDA_MANAGED_MEMORY"] = "0"  # Forcing in-memory engine to not use UVM, just for the demo
gpu_in_mem_engine = pl.GPUEngine(executor="in-memory", raise_on_fail=True)

In [ ]:
%%time
taxi_summary(df_green, complex_keys).collect(engine=gpu_in_mem_engine)

In [ ]:
result, prof = taxi_summary(df_green, complex_keys).profile(engine=gpu_in_mem_engine)
prof.with_columns(((pl.col("end") - pl.col("start")) / pl.col("end").max() * 100).alias("pct_time_spent"))

## Streaming for a single GPU

Polars' CPU streaming engine is stable since 1.41, and since polars 2.0 it is what `engine="auto"` resolves to for lazy queries. This notebook installs polars 1.44.2 for GPU compatibility, where `"auto"` still means in-memory, so CPU cells pass `engine="streaming"` explicitly. On GPU the streaming executor is the default behind `engine="gpu"`; it processes partitions and can spill buffers to host memory. This is separate from CUDA managed memory (UVM), used by the historical in-memory examples linked below.

[UVM, data larger than GPU memory post (NVIDIA blog)](https://developer.nvidia.com/blog/how-to-work-with-data-exceeding-vram-in-the-polars-gpu-engine/)

[UVM, data larger than GPU memory post (Polars blog)](https://pola.rs/posts/uvm-larger-than-ram-gpu/)

[Polars GPU Streaming post (NVIDIA blog)](https://developer.nvidia.com/blog/rapids-adds-gpu-polars-streaming-a-unified-gnn-api-and-zero-code-ml-speedups)


In [ ]:
%%time
taxi_summary(df, complex_keys).collect(
    engine=gpu_in_mem_engine
)  # RuntimeError: parallel_for failed: cudaErrorMemoryAllocation: out of memory

In [ ]:
# After an out-of-memory error, restart the kernel/runtime manually, then rerun setup cell 3 before engine cell 20.

In [ ]:
# Streaming uses partitioned execution and host spilling, not the historical manual RMM UVM setup.
gpu_streaming_engine = pl.GPUEngine(executor="streaming", raise_on_fail=True)

In [ ]:
%%time
taxi_summary(df, complex_keys).collect(engine=gpu_streaming_engine)

In [ ]:
# Stop here on a single-GPU runtime. The optional section below releases the implicit streaming engine before Dask.

## Multi GPU Streaming. Currently in experimental stage.
#### NOTE: Only run this part in an environment where are multiple GPUs. Kaggle can be a free option to test it.
After installing the Dask extras, restart the kernel/runtime if packages were already imported, then rerun setup cell 3 and cells 25–26.

In [ ]:
%pip install "polars[gpu]==1.44.2" "cudf-polars-cu12[dask]>=26.10.2"

In [ ]:
# DaskEngine starts one worker per visible GPU; the final cell owns its lifetime.
from cudf_polars.engine.dask import DaskEngine
from cudf_polars.engine.default_singleton_engine import DefaultSingletonEngine

In [ ]:
%%time
DefaultSingletonEngine.shutdown()  # The implicit single-GPU streaming engine cannot coexist with DaskEngine.
with DaskEngine(engine_options={"raise_on_fail": True}) as multi_gpu_streaming:
    result = taxi_summary(df, complex_keys).collect(engine=multi_gpu_streaming)
result